# Multi-stock next-day close prediction

The original notebook split by **row index after sorting by ticker**, so entire companies leaked into train or test. This version splits on **calendar dates** shared across all stocks, then trains LightGBM / XGBoost / CatBoost / Ridge.

In [ ]:
# %pip install -q pandas numpy scikit-learn xgboost lightgbm catboost matplotlib

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path.cwd()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "src" / "features.py").exists():
        sys.path.insert(0, str(candidate))
        break

from src.features import FEATURE_COLUMNS, add_grouped_features
from src.metrics import regression_metrics
from src.models import build_tabular_models
from src.split import global_date_split

In [ ]:
def load_cleaned():
    for path in [
        Path("data/Cleaned_DSE_Data.csv"),
        Path("../data/Cleaned_DSE_Data.csv"),
        Path("Cleaned.csv"),
        Path("Cleaned_DSE_Data.csv"),
    ]:
        if path.exists():
            print("Loaded", path)
            return pd.read_csv(path, parse_dates=["Date"])
    raise FileNotFoundError("Run the cleaning notebook first.")


df = load_cleaned().sort_values(["Trading_Code", "Date"]).reset_index(drop=True)

# Keep liquid names so training stays tractable
counts = df.groupby("Trading_Code").size()
keep = counts[counts >= 800].index
df = df[df["Trading_Code"].isin(keep)].copy()
print("Stocks:", df["Trading_Code"].nunique(), "rows:", len(df))

df = add_grouped_features(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

le = LabelEncoder()
df["Stock_ID"] = le.fit_transform(df["Trading_Code"])
feature_cols = [c for c in FEATURE_COLUMNS if c in df.columns] + ["Stock_ID"]
print("Features:", len(feature_cols))

In [ ]:
train_df, val_df, test_df, train_cut, val_cut = global_date_split(df, train_ratio=0.70, val_ratio=0.15)
print("Train ≤", train_cut.date(), "| Val ≤", val_cut.date(), "| Test after that")
print(len(train_df), len(val_df), len(test_df))

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])
y_train = train_df["Target_Close"].to_numpy()
y_val = val_df["Target_Close"].to_numpy()
y_test = test_df["Target_Close"].to_numpy()
close_test = test_df["Close"].to_numpy()

In [ ]:
wanted = ["Ridge", "XGBoost", "LightGBM", "CatBoost", "HistGBM"]
models = {k: v for k, v in build_tabular_models().items() if k in wanted}

rows, preds = [], {}
for name, model in models.items():
    fit_kwargs = {}
    if name == "XGBoost":
        fit_kwargs = {"eval_set": [(X_val, y_val)], "verbose": False}
    elif name == "LightGBM":
        try:
            from lightgbm import early_stopping
            fit_kwargs = {"eval_set": [(X_val, y_val)], "callbacks": [early_stopping(60, verbose=False)]}
        except Exception:
            fit_kwargs = {"eval_set": [(X_val, y_val)]}
    elif name == "CatBoost":
        fit_kwargs = {"eval_set": (X_val, y_val)}

    try:
        model.fit(X_train, y_train, **fit_kwargs)
    except TypeError:
        model.fit(X_train, y_train)

    pred = np.asarray(model.predict(X_test)).ravel()
    preds[name] = pred
    metrics = regression_metrics(y_test, pred, close_today=close_test)
    metrics["Model"] = name
    rows.append(metrics)
    print(name, {k: round(v, 4) for k, v in metrics.items() if k != "Model"})

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(results)

In [ ]:
top = results.nsmallest(3, "RMSE")["Model"].tolist()
w = 1.0 / results.set_index("Model").loc[top, "RMSE"]
w = w / w.sum()
ensemble = sum(w[m] * preds[m] for m in top)
ens = regression_metrics(y_test, ensemble, close_today=close_test)
ens["Model"] = "Ensemble"
results = pd.concat([results, pd.DataFrame([ens])], ignore_index=True).sort_values("RMSE")
display(results.round(4))

best = results.iloc[0]["Model"]
test_df = test_df.copy()
test_df["Predicted_Close"] = ensemble if best == "Ensemble" else preds[best]

In [ ]:
ticker = "BEXIMCO" if "BEXIMCO" in set(test_df["Trading_Code"]) else test_df["Trading_Code"].iloc[0]
plot_df = test_df[test_df["Trading_Code"] == ticker].sort_values("Date")

plt.figure(figsize=(12, 5))
plt.plot(plot_df["Date"], plot_df["Target_Close"], label="Actual next close")
plt.plot(plot_df["Date"], plot_df["Predicted_Close"], label=f"{best} predicted")
plt.title(f"{ticker} hold-out period")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plot_df[["Date", "Trading_Code", "Close", "Target_Close", "Predicted_Close"]].head(15)